# Conditional Simulation Compute Task

This notebook demonstrates how to run conditional simulation (consim) compute tasks using the `evo-compute` package.

Conditional simulation is a geostatistical technique that generates multiple equally probable realizations
of a spatial variable, honoring both the conditioning data and the spatial correlation model (variogram).
Unlike kriging which produces a single "best estimate", simulation captures the full range of uncertainty.

## Authentication

First, authenticate using the `ServiceManagerWidget`:

In [ ]:
from evo.notebooks import ServiceManagerWidget

# Use integration environment
manager = await ServiceManagerWidget.with_auth_code(
    client_id="<my-client-id>",
    cache_location="./notebook-data",
).login()

In [ ]:
# Load the widgets extension for rich HTML display
%load_ext evo.widgets

## Example 1: Run Conditional Simulation on Existing Objects

This example shows how to run conditional simulation using existing geoscience objects (source pointset, target grid, and variogram).

### Load the Source PointSet, Target Grid, and Variogram

In [ ]:
from evo.objects.typed import object_from_path

# Load objects by path
source_pointset = await object_from_path(manager, "source_pointset.json")
target_grid = await object_from_path(manager, "target_grid.json")
variogram = await object_from_path(manager, "variogram.json")

In [ ]:
# Pretty-print the source pointset (includes Portal/Viewer links)
source_pointset

In [ ]:
# View the source pointset attributes
source_pointset.attributes

In [ ]:
variogram

In [ ]:
# Get source data statistics for distribution setup
source_df = await source_pointset.to_dataframe()
source_attr = "Ag_ppm Values"  # Replace with your attribute name

print(f"Source attribute: {source_attr}")
print(f"  Min:  {source_df[source_attr].min():.4f}")
print(f"  Max:  {source_df[source_attr].max():.4f}")
print(f"  Mean: {source_df[source_attr].mean():.4f}")
print(f"  Std:  {source_df[source_attr].std():.4f}")

### Optionally, Define the Distribution for Normal-Score Transformation

Conditional simulation requires transforming data to normal-score space, if not specified by the user it will be constructed by deault. Explicitly creating the distrubtion provides more control over tail extrapolation.

In [ ]:
from evo.compute.tasks.simulation import Distribution, LowerTail, TailExtrapolation, UpperTail

# Define tail extrapolation based on data statistics
# Upper tail max should be > data max
# Lower tail min should be < data min (or 0 for grades that can't be negative)

distribution = Distribution(
    tail_extrapolation=TailExtrapolation(
        upper=UpperTail(
            power=0.5,  # Controls tail shape (0.5 = linear extrapolation)
            max=source_df[source_attr].max() * 1.5,  # 50% beyond max
        ),
        lower=LowerTail(
            power=0.5,
            min=0.0,  # Grades can't be negative
        ),
    ),
)

print("Distribution configured:")
print(
    f"  Upper tail: power={distribution.tail_extrapolation.upper.power}, max={distribution.tail_extrapolation.upper.max}"
)
print(
    f"  Lower tail: power={distribution.tail_extrapolation.lower.power}, min={distribution.tail_extrapolation.lower.min}"
)

### Run Conditional Simulation

In [ ]:
from evo.compute.tasks import SearchNeighborhood, run
from evo.compute.tasks.simulation import ConsimParameters

# Get ellipsoid from the variogram structure with largest range (default)
var_ell = variogram.get_ellipsoid()

# Create search ellipsoid by scaling the variogram ellipsoid by 2x
search_ellipsoid = var_ell.scaled(2.0)

In [ ]:
# Create simulation parameters
consim_params = ConsimParameters(
    # Source and target
    source=source_pointset.attributes[source_attr],
    target_object=target_grid,
    variogram=variogram,
    # Search neighborhood
    search=SearchNeighborhood(
        ellipsoid=search_ellipsoid,
        max_samples=20,
    ),
    # Distribution for normal-score transformation
    distribution=distribution,
    # Simulation settings
    number_of_simulations=100,  # Total realizations to generate
    number_of_simulations_to_save=5,  # How many to save as attributes
    number_of_lines=500,  # Turning bands lines
    random_seed=12345,  # For reproducibility
    # Output quantiles (e.g., P10, P50, P90)
    location_wise_quantiles=[0.1, 0.5, 0.9],
)

# Run the simulation task (progress feedback is shown by default)
print("Submitting conditional simulation task...")
result = await run(manager, consim_params, preview=True)

# Display the result summary
print(result)

In [ ]:
# Access summary statistics
print(f"Mean attribute: {result.mean_attribute.name}")
print(f"Variance attribute: {result.variance_attribute.name}")
print(f"Min attribute: {result.min_attribute.name}")
print(f"Max attribute: {result.max_attribute.name}")

# List quantile attributes
print("\nQuantile attributes:")
for q in result.quantile_attributes:
    print(f"  P{int(q.quantile * 100)}: {q.name}")

In [ ]:
# Get the simulation results as a DataFrame
df = await result.to_dataframe()
print(f"Retrieved {len(df)} cells with simulation results")
df.head()

In [ ]:
# Visualize uncertainty (P90 - P10 spread)
import plotly.express as px

# Find quantile column names
p10_col = [q.name for q in result.quantile_attributes if q.quantile == 0.1][0]
p50_col = [q.name for q in result.quantile_attributes if q.quantile == 0.5][0]
p90_col = [q.name for q in result.quantile_attributes if q.quantile == 0.9][0]

df["uncertainty"] = df[p90_col] - df[p10_col]

fig = px.histogram(df, x="uncertainty", nbins=50, title="Uncertainty Distribution (P90 - P10)")
fig.show()

## Example 2: Run Simulation with Validation

This example demonstrates how to run conditional simulation with validation enabled, which generates a validation report and provides a link to the validation dashboard.

In [ ]:
from evo.compute.tasks.simulation import (
    ConsimParameters,
    ReportMeanThresholds,
    ValidationReportContext,
    ValidationReportContextItem,
)

# Create simulation parameters with validation
consim_params_validated = ConsimParameters(
    # Same source/target/variogram as before
    source=source_pointset.attributes[source_attr],
    target_object=target_grid,
    variogram=variogram,
    # Search and distribution
    search=SearchNeighborhood(
        ellipsoid=search_ellipsoid,
        max_samples=20,
    ),
    distribution=distribution,
    # Simulation settings
    number_of_simulations=100,
    number_of_simulations_to_save=5,
    location_wise_quantiles=[0.1, 0.25, 0.5, 0.75, 0.9],
    # Enable validation
    perform_validation=True,
    # Validation report context
    report_context=ValidationReportContext(
        title="Grade Simulation Validation Report",
        details=[
            ValidationReportContextItem(label="Domain", value="All"),
            ValidationReportContextItem(label="Variable", value=source_attr),
            ValidationReportContextItem(label="Method", value="Turning Bands"),
        ],
    ),
    # User defined mean bias thresholds
    report_mean_thresholds=ReportMeanThresholds(
        acceptable=5.0,  # <5% difference is acceptable
        marginal=10.0,  # 5-10% is marginal, >10% is unacceptable
    ),
)

print("Submitting simulation with validation...")
validated_result = await run(manager, consim_params_validated, preview=True)

print(validated_result)

In [ ]:
# Check validation summary
if validated_result.validation_summary:
    vs = validated_result.validation_summary
    pct_diff = abs(vs.mean - vs.reference_mean) / vs.reference_mean * 100
    print("Validation Summary:")
    print(f"  Reference mean (input):  {vs.reference_mean:.4f}")
    print(f"  Simulated mean:          {vs.mean:.4f}")
    print(f"  Difference:              {pct_diff:.2f}%")
else:
    print("No validation summary available")

# Check for validation report link
if validated_result.validation_report:
    print(f"\nValidation report: {validated_result.validation_report.name}")
    print(f"  Reference: {validated_result.validation_report.reference}")

# Check for dashboard link
if validated_result.dashboard_link:
    print(f"\nDashboard: {validated_result.dashboard_link}")

## Example 3: Create Objects and Run Simulation

This example shows how to create the input objects from scratch and then run conditional simulation.

### Create the Source PointSet

In [ ]:
import uuid

import numpy as np
import pandas as pd
from evo.objects.typed import EpsgCode, PointSet, PointSetData

# Generate sample point data with realistic grade distribution
n_points = 500
np.random.seed(42)

# Create points in a 1000x1000x200 domain
x = np.random.uniform(0, 1000, n_points)
y = np.random.uniform(0, 1000, n_points)
z = np.random.uniform(0, 200, n_points)

# Create a grade attribute with lognormal distribution (common for mineral grades)
grade = np.random.lognormal(mean=0.5, sigma=0.8, size=n_points)
grade = np.clip(grade, 0.01, 20)  # Clip to realistic range

# Create pointset using PointSetData
pointset_data = PointSetData(
    name=f"Simulation Source Points - {uuid.uuid4()}",
    coordinate_reference_system=EpsgCode(32632),
    locations=pd.DataFrame({"x": x, "y": y, "z": z, "grade": grade}),
)

# Create the pointset object
source_pointset_created = await PointSet.create(manager, pointset_data)

print(f"Created source pointset: {source_pointset_created.name}")
print("Grade statistics:")
print(f"  Min:  {grade.min():.4f}")
print(f"  Max:  {grade.max():.4f}")
print(f"  Mean: {grade.mean():.4f}")

In [ ]:
# Pretty-print the created pointset
source_pointset_created

### Create a Variogram

In [ ]:
from evo.objects.typed import (
    Ellipsoid,
    EllipsoidRanges,
    Rotation,
    SphericalStructure,
    Variogram,
    VariogramData,
)

# Define a nested spherical variogram model
variogram_data = VariogramData(
    name=f"Simulation Variogram - {uuid.uuid4()}",
    sill=1.0,
    nugget=0.1,
    is_rotation_fixed=True,
    modelling_space="data",
    data_variance=1.0,
    structures=[
        # Short-range structure
        SphericalStructure(
            contribution=0.4,
            anisotropy=Ellipsoid(
                ranges=EllipsoidRanges(major=100.0, semi_major=80.0, minor=50.0),
                rotation=Rotation(dip_azimuth=45.0, dip=0.0, pitch=0.0),
            ),
        ),
        # Long-range structure
        SphericalStructure(
            contribution=0.5,
            anisotropy=Ellipsoid(
                ranges=EllipsoidRanges(major=300.0, semi_major=200.0, minor=100.0),
                rotation=Rotation(dip_azimuth=45.0, dip=0.0, pitch=0.0),
            ),
        ),
    ],
    attribute="grade",
    domain="all",
)

variogram_created = await Variogram.create(manager, variogram_data)

print(f"Created variogram: {variogram_created.name}")

In [ ]:
# Pretty-print the created variogram
variogram_created

### Create the Target Grid

In [ ]:
from evo.objects.typed import Point3, RegularMasked3DGrid, RegularMasked3DGridData, Size3d, Size3i
from evo.objects.typed import Rotation as GridRotation

# Define grid dimensions
nx, ny, nz = 20, 20, 10  # Number of cells in each direction
cell_size = 50.0  # Size of each cell

# Create a mask for the grid (all cells active)
total_cells = nx * ny * nz
mask = np.ones(total_cells, dtype=bool)

# Create masked grid
grid_data = RegularMasked3DGridData(
    name=f"Simulation Target Grid - {uuid.uuid4()}",
    coordinate_reference_system=EpsgCode(32632),
    origin=Point3(0, 0, 0),
    size=Size3i(nx, ny, nz),
    cell_size=Size3d(cell_size, cell_size, cell_size / 2),  # Smaller vertical cells
    rotation=GridRotation(0, 0, 0),
    mask=mask,
    cell_data=None,
)

target_grid_created = await RegularMasked3DGrid.create(manager, grid_data)

print(f"Created target grid: {target_grid_created.name}")
print(f"  Total cells: {nx} x {ny} x {nz} = {total_cells}")

In [ ]:
# Pretty-print the created grid
target_grid_created

### Run Conditional Simulation on Created Objects

In [ ]:
from evo.compute.tasks import SearchNeighborhood, run
from evo.compute.tasks.simulation import (
    ConsimParameters,
    Distribution,
    LowerTail,
    TailExtrapolation,
    UpperTail,
)

# Get ellipsoid from the variogram (longest range structure)
var_ell = variogram_created.get_ellipsoid()
search_ellipsoid = var_ell.scaled(2.0)

# Define distribution based on data
distribution_created = Distribution(
    tail_extrapolation=TailExtrapolation(
        upper=UpperTail(power=0.5, max=grade.max() * 1.5),
        lower=LowerTail(power=0.5, min=0.0),
    ),
)

# Create simulation parameters
consim_params_created = ConsimParameters(
    source=source_pointset_created.locations.attributes["grade"],
    target_object=target_grid_created,
    variogram=variogram_created,
    search=SearchNeighborhood(
        ellipsoid=search_ellipsoid,
        max_samples=30,
    ),
    distribution=distribution_created,
    number_of_simulations=50,
    number_of_simulations_to_save=3,
    location_wise_quantiles=[0.1, 0.5, 0.9],
    random_seed=42,
)

print("Submitting conditional simulation task...")
result_created = await run(manager, consim_params_created, preview=True)

print("Task completed!")
print(result_created)

In [ ]:
# Get the simulation results as a DataFrame
df_created = await result_created.to_dataframe()
print(f"Retrieved {len(df_created)} cells with simulation results")
df_created.head()

In [ ]:
# Load the target object for further analysis
target_with_results = await result_created.get_target_object()
target_with_results